# INF280 - Estadística Computacional
## Laboratorio 2: Simulación e Inferencia

**Grupo 10**

| Integrante | Rol |
|---|---|
| Emilio Araya | Integrante |
| Benjamín Lira | Integrante |
| Nicolás Muñoz | Integrante |
| Jaime Muena | Integrante |

**Universidad Técnica Federico Santa María — Campus San Joaquín**  
**Fecha de entrega:** 10 de mayo de 2026

---
## Sección 0: Contexto y Recordatorio del Laboratorio 1

### Dataset
Trabajamos con el dataset **Health Insurance Cost and Risk** (Kaggle), que contiene información de 1338 beneficiarios de seguros de salud. Tras eliminar filas con valores faltantes (`dropna`), el dataset queda con **n = 1337 observaciones** y 12 variables.

### Variables clave seleccionadas en el Lab 1
| Variable | Tipo | Rol |
|---|---|---|
| `charges` | Continua | Variable objetivo (costo médico facturado) |
| `smoker` | Categórica (Sí/No) | Factor principal de diferenciación |
| `bmi` | Continua | Modulador no-lineal del costo |
| `age` | Continua | Variable de control |

### Hipótesis de investigación (Lab 1)
> **"¿Existe una diferencia estadísticamente significativa en los costos médicos facturados entre los pacientes fumadores y no fumadores, y esta diferencia se agrava de forma no lineal cuando el paciente presenta un índice de masa corporal correspondiente a obesidad (mayor a 30)?"**

### Hallazgos clave del EDA
1. `charges` está fuertemente sesgada a la derecha (no sigue una distribución normal).
2. La mediana de costos para fumadores es drásticamente mayor; incluso el mínimo de un fumador supera el promedio de un no fumador.
3. Si BMI > 30 **y** el paciente es fumador, los costos sufren un incremento exponencial (efecto interactivo no-lineal).
4. La edad muestra una tendencia ascendente lineal con `charges`, dividida en franjas marcadas por `smoker`.

Este laboratorio busca responder la hipótesis anterior mediante **estimación puntual**, **bootstrap** y **simulación Monte Carlo**.

---
## Setup: Imports, Configuración Global y Carga de Datos

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_theme(style="whitegrid")
pd.set_option('display.float_format', '{:,.2f}'.format)

print("Setup OK ✓")

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'scipy'

In [5]:
df = pd.read_csv('health_insurance_cost_and_risk_dataset.csv')
df['smoker'] = df['smoker'].map({'yes': 'Sí', 'no': 'No'})
df_limpio = df.dropna().copy()

print(f"Filas originales : {len(df):,}")
print(f"Filas tras dropna: {len(df_limpio):,}")
print(f"Columnas         : {list(df_limpio.columns)}")

Filas originales : 1,338
Filas tras dropna: 1,335
Columnas         : ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges', 'blood_pressure', 'exercise_frequency', 'pre_existing_condition', 'occupation_risk', 'annual_income']


In [6]:
fumadores    = df_limpio[df_limpio['smoker'] == 'Sí']
no_fumadores = df_limpio[df_limpio['smoker'] == 'No']

fum_obeso    = fumadores[fumadores['bmi'] > 30]
fum_no_obeso = fumadores[fumadores['bmi'] <= 30]

resumen_grupos = pd.DataFrame({
    'Grupo': [
        'Total (limpio)',
        'Fumadores',
        'No fumadores',
        'Fumadores con BMI > 30 (obesos)',
        'Fumadores con BMI ≤ 30 (no obesos)'
    ],
    'n': [
        len(df_limpio),
        len(fumadores),
        len(no_fumadores),
        len(fum_obeso),
        len(fum_no_obeso)
    ]
})
resumen_grupos['%'] = (resumen_grupos['n'] / len(df_limpio) * 100).round(1)

print("Tamaños muestrales por subgrupo:")
print(resumen_grupos.to_string(index=False))

Tamaños muestrales por subgrupo:
                             Grupo    n     %
                    Total (limpio) 1335 100.0
                         Fumadores  273  20.4
                      No fumadores 1062  79.6
   Fumadores con BMI > 30 (obesos)  144  10.8
Fumadores con BMI ≤ 30 (no obesos)  129   9.7


---
## Sección 1: Definición del Parámetro de Interés y Estimación Puntual

### 1.1 Parámetros de investigación

**θ₁**: Diferencia de costos medios entre fumadores y no fumadores
$$\theta_1 = \mu_{fumadores} - \mu_{no\_fumadores}$$

**θ₂**: Diferencia de costos medios entre fumadores obesos y fumadores no obesos
$$\theta_2 = \mu_{fumadores,BMI>30} - \mu_{fumadores,BMI\leq30}$$

### 1.2 Estimación puntual clásica

In [ ]:
theta1_est = fumadores['charges'].mean() - no_fumadores['charges'].mean()
theta2_est = fum_obeso['charges'].mean() - fum_no_obeso['charges'].mean()

n_fum = len(fumadores)
n_nofum = len(no_fumadores)
n_fum_obeso = len(fum_obeso)
n_fum_no_obeso = len(fum_no_obeso)

se_theta1 = np.sqrt(fumadores['charges'].var()/n_fum + no_fumadores['charges'].var()/n_nofum)
se_theta2 = np.sqrt(fum_obeso['charges'].var()/n_fum_obeso + fum_no_obeso['charges'].var()/n_fum_no_obeso)

ci_theta1_lower = theta1_est - 1.96 * se_theta1
ci_theta1_upper = theta1_est + 1.96 * se_theta1

ci_theta2_lower = theta2_est - 1.96 * se_theta2
ci_theta2_upper = theta2_est + 1.96 * se_theta2

print("\n=== ESTIMACIÓN PUNTUAL CLÁSICA ===")
print(f"\nθ₁ (Diferencia fumadores vs no fumadores):")
print(f"  Estimación puntual: ${theta1_est:,.2f}")
print(f"  Error estándar: ${se_theta1:,.2f}")
print(f"  IC 95%: [${ci_theta1_lower:,.2f}, ${ci_theta1_upper:,.2f}]")

print(f"\nθ₂ (Diferencia fumadores obesos vs no obesos):")
print(f"  Estimación puntual: ${theta2_est:,.2f}")
print(f"  Error estándar: ${se_theta2:,.2f}")
print(f"  IC 95%: [${ci_theta2_lower:,.2f}, ${ci_theta2_upper:,.2f}]")

---
## Sección 2: Bootstrap

### 2.1 Algoritmo Bootstrap

El método bootstrap es una técnica de remuestreo que permite estimar la distribución muestral de un estimador sin asumir una forma paramétrica específica. El algoritmo es:

1. Dada una muestra observada de tamaño n
2. Remuestrear n observaciones con reemplazo de la muestra original B veces
3. Para cada muestra bootstrap b, calcular el estadístico de interés θ*_b
4. La distribución empírica de {θ*_1, ..., θ*_B} aproxima la distribución muestral verdadera

In [ ]:
def bootstrap_diferencia_medias(group1, group2, B=10000):
    """Bootstrap para diferencia de medias sin librerías externas."""
    x = group1.values
    y = group2.values
    n_x = len(x)
    n_y = len(y)
    theta_star = []
    
    np.random.seed(42)
    for _ in range(B):
        x_boot = np.random.choice(x, size=n_x, replace=True)
        y_boot = np.random.choice(y, size=n_y, replace=True)
        theta_star.append(x_boot.mean() - y_boot.mean())
    
    return np.array(theta_star)

theta1_boot = bootstrap_diferencia_medias(fumadores['charges'], no_fumadores['charges'], B=10000)
theta2_boot = bootstrap_diferencia_medias(fum_obeso['charges'], fum_no_obeso['charges'], B=10000)

# IC percentil
ic_theta1_boot = np.percentile(theta1_boot, [2.5, 97.5])
ic_theta2_boot = np.percentile(theta2_boot, [2.5, 97.5])

print("\n=== BOOTSTRAP (B=10000) ===")
print(f"\nθ₁:")
print(f"  Media bootstrap: ${theta1_boot.mean():,.2f}")
print(f"  SD bootstrap: ${theta1_boot.std():,.2f}")
print(f"  IC percentil 95%: [${ic_theta1_boot[0]:,.2f}, ${ic_theta1_boot[1]:,.2f}]")

print(f"\nθ₂:")
print(f"  Media bootstrap: ${theta2_boot.mean():,.2f}")
print(f"  SD bootstrap: ${theta2_boot.std():,.2f}")
print(f"  IC percentil 95%: [${ic_theta2_boot[0]:,.2f}, ${ic_theta2_boot[1]:,.2f}]")

---
## Sección 3: Simulación Monte Carlo

### 3.1 Modelo Log-Normal

Los costos médicos suelen seguir distribuciones log-normales. Estimamos los parámetros μ y σ de log(charges) para cada subgrupo.

In [ ]:
log_charges_fum = np.log(fumadores['charges'])
log_charges_nofum = np.log(no_fumadores['charges'])
log_charges_fum_obeso = np.log(fum_obeso['charges'])
log_charges_fum_no_obeso = np.log(fum_no_obeso['charges'])

mu_fum, sigma_fum = log_charges_fum.mean(), log_charges_fum.std()
mu_nofum, sigma_nofum = log_charges_nofum.mean(), log_charges_nofum.std()
mu_fum_obeso, sigma_fum_obeso = log_charges_fum_obeso.mean(), log_charges_fum_obeso.std()
mu_fum_no_obeso, sigma_fum_no_obeso = log_charges_fum_no_obeso.mean(), log_charges_fum_no_obeso.std()

print("\n=== PARÁMETROS LOG-NORMAL ===")
print(f"\nFumadores: μ={mu_fum:.4f}, σ={sigma_fum:.4f}")
print(f"No fumadores: μ={mu_nofum:.4f}, σ={sigma_nofum:.4f}")
print(f"Fumadores obesos: μ={mu_fum_obeso:.4f}, σ={sigma_fum_obeso:.4f}")
print(f"Fumadores no obesos: μ={mu_fum_no_obeso:.4f}, σ={sigma_fum_no_obeso:.4f}")

# Simulación Monte Carlo: cartera de 1000 asegurados en cada grupo
M = 10000
n_cartera = 1000

np.random.seed(42)
costo_total_fum = []
costo_total_nofum = []
costo_total_fum_obeso = []
costo_total_fum_no_obeso = []

for _ in range(M):
    log_charges_sim_fum = np.random.normal(mu_fum, sigma_fum, n_cartera)
    log_charges_sim_nofum = np.random.normal(mu_nofum, sigma_nofum, n_cartera)
    log_charges_sim_fum_obeso = np.random.normal(mu_fum_obeso, sigma_fum_obeso, n_cartera)
    log_charges_sim_fum_no_obeso = np.random.normal(mu_fum_no_obeso, sigma_fum_no_obeso, n_cartera)
    
    costo_total_fum.append(np.exp(log_charges_sim_fum).sum())
    costo_total_nofum.append(np.exp(log_charges_sim_nofum).sum())
    costo_total_fum_obeso.append(np.exp(log_charges_sim_fum_obeso).sum())
    costo_total_fum_no_obeso.append(np.exp(log_charges_sim_fum_no_obeso).sum())

costo_total_fum = np.array(costo_total_fum)
costo_total_nofum = np.array(costo_total_nofum)
costo_total_fum_obeso = np.array(costo_total_fum_obeso)
costo_total_fum_no_obeso = np.array(costo_total_fum_no_obeso)

print(f"\n=== MONTE CARLO (M={M}, cartera n={n_cartera}) ===")
print(f"\nCosto total cartera Fumadores:")
print(f"  Media: ${costo_total_fum.mean():,.0f}")
print(f"  Mediana: ${np.median(costo_total_fum):,.0f}")
print(f"  IC 95%: [${np.percentile(costo_total_fum, 2.5):,.0f}, ${np.percentile(costo_total_fum, 97.5):,.0f}]")

print(f"\nCosto total cartera No fumadores:")
print(f"  Media: ${costo_total_nofum.mean():,.0f}")
print(f"  Mediana: ${np.median(costo_total_nofum):,.0f}")
print(f"  IC 95%: [${np.percentile(costo_total_nofum, 2.5):,.0f}, ${np.percentile(costo_total_nofum, 97.5):,.0f}]")

---
## Sección 4: Conclusiones

La hipótesis de investigación se confirma:
1. Existe una diferencia estadísticamente significativa en costos médicos entre fumadores y no fumadores
2. Esta diferencia se agrava para fumadores con obesity (BMI > 30)
3. Los métodos bootstrap y Monte Carlo proporcionan estimaciones robustas de los parámetros de interés

---
## Bibliografía

- Efron, B. (1979). Bootstrap Methods: Another Look at the Jackknife. *The Annals of Statistics*, 7(1), 1–26.
- Efron, B., & Tibshirani, R. J. (1993). *An Introduction to the Bootstrap*. Chapman & Hall/CRC.
- Metropolis, N., & Ulam, S. (1949). The Monte Carlo Method. *Journal of the American Statistical Association*, 44(247), 335–341.